# Week 7 — Breast Cancer Diagnosis Evaluator

**Theme:** Model evaluation — cross-validation & evaluation metrics

So far we've measured models with a single train/test split and a single
accuracy number. That's not enough for a real decision — especially in a
medical context, where the *kind* of mistake matters a lot: missing a
malignant tumor (false negative) is far worse than an unnecessary follow-up
test (false positive).

**Dataset:** scikit-learn's built-in breast cancer dataset — 569 patients,
30 measurements from a tumor scan, label = malignant or benign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score, precision_recall_curve,
)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Classes:", dict(zip(data.target_names, [0, 1])))
print("Shape:", X.shape)
print("Class balance:", np.bincount(y), "(0=malignant, 1=benign)")

## 1. A single train/test split — and why it's not enough

We build a `Pipeline` so scaling always happens *inside* the same fit/predict
step (never fit the scaler on test data — that would leak information).

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)
model.fit(X_train, y_train)
single_split_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy on THIS ONE split: {single_split_acc:.2%}")

In [ ]:
# What if we'd used a different random split? Try several seeds.
accs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
    accs.append(accuracy_score(yte, m.predict(Xte)))

print("Accuracy across 10 different random splits:", [f"{a:.2%}" for a in accs])
print(f"Range: {min(accs):.2%} to {max(accs):.2%} — a single split can be lucky or unlucky!")

## 2. k-fold cross-validation: a more honest estimate

Instead of one split, **k-fold cross-validation** splits the data into `k`
parts, trains on `k-1` of them and tests on the last, and rotates — so every
example gets used for testing exactly once. We report the average.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("5-fold CV accuracy per fold:", [f"{s:.2%}" for s in cv_scores])
print(f"Mean: {cv_scores.mean():.2%}  (+/- {cv_scores.std():.2%})")

## 3. Confusion matrix: what kind of mistakes does it make?

Accuracy hides *which* mistakes happen. The confusion matrix doesn't.

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. Precision, recall, and why accuracy alone can mislead

- **Precision** — of everything we predicted "benign", how much really was benign?
- **Recall** — of everything that really was "malignant", how much did we catch?

In this task, missing a malignant tumor (a false negative for the malignant
class) is the costly mistake — so **recall on the malignant class** matters a
lot, sometimes more than overall accuracy.

In [ ]:
# malignant is class 0 in this dataset — let's look at recall for it specifically
report = classification_report(y_test, y_pred, target_names=data.target_names, output_dict=True)
print(f"Recall on malignant cases: {report['malignant']['recall']:.2%}")
print(f"Precision on malignant cases: {report['malignant']['precision']:.2%}")

## 5. ROC curve and AUC

The model doesn't just output a class — it outputs a *probability*. The ROC
curve shows the tradeoff between catching more true positives (recall) and
generating more false alarms, as we sweep the decision threshold.

In [ ]:
y_scores = model.predict_proba(X_test)[:, 1]  # probability of "benign"
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Try it yourself

1. **Move the decision threshold.** The default threshold is 0.5. Try
   `y_pred_strict = (y_scores >= 0.9).astype(int)` (only call it "benign" if
   90%+ confident) and recompute the confusion matrix — how do precision and
   recall for "malignant" change?
2. **Precision-recall curve.** Use `precision_recall_curve(y_test, y_scores)`
   to plot precision vs. recall directly — when might this be more informative
   than the ROC curve? (Hint: think about class imbalance.)
3. **10-fold vs. 5-fold.** Re-run cross-validation with `n_splits=10` — does
   the mean accuracy change much? Does the standard deviation?
4. **A worse model on purpose.** Train a `LogisticRegression` with only 2 of
   the 30 features — how much does accuracy, recall, and AUC all drop?

---
## 🎯 캡스톤: 과제 마감 지각 위험 예측기

가상의 과제 제출 기록 250건("마감 며칠 전에 시작했는지 / 예상 소요시간 / 그 주 다른 마감 개수 / 난이도" -> "지각 여부")을 드립니다. 위에서 배운 **로지스틱 회귀 + 교차검증 + confusion matrix/정밀도·재현율**을 그대로 적용해서, 여러분의 다음 과제가 늦을 위험이 얼마나 되는지 예측하는 모델을 만들어보세요.

**확장 아이디어:** 이번 학기 실제 과제들의 (착수 시점, 예상 소요시간, 그 주 다른 과제 수, 난이도, 실제 지각 여부)를 기록해서 `assignments_df`를 바꾸면, 진짜 "내 과제 지각 위험 알리미"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(9)
n_assignments = 250

days_before_started = np.clip(rng.exponential(3, n_assignments), 0, 14)
estimated_hours = np.clip(rng.normal(6, 3, n_assignments), 0.5, 20)
num_other_deadlines = rng.integers(0, 5, n_assignments)
difficulty = rng.integers(1, 6, n_assignments)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

logit = (
    -1.0
    - 0.55 * days_before_started
    + 0.22 * estimated_hours
    + 0.5 * num_other_deadlines
    + 0.35 * difficulty
)
late_prob = sigmoid(logit)
submitted_late = (rng.random(n_assignments) < late_prob).astype(int)

assignments_df = pd.DataFrame({
    "days_before_started": days_before_started.round(1),
    "estimated_hours": estimated_hours.round(1),
    "num_other_deadlines": num_other_deadlines,
    "difficulty": difficulty,
    "submitted_late": submitted_late,
})
print("전체 지각 비율:", assignments_df["submitted_late"].mean().round(2))
assignments_df.head()

### 여러분의 과제

1. `days_before_started`, `estimated_hours`, `num_other_deadlines`, `difficulty`를 입력(X), `submitted_late`를 정답(y)으로 하여 `StandardScaler` + `LogisticRegression` 파이프라인을 학습시키세요. (Week 7 본문 코드 참고)
2. `cross_val_score`로 5-fold 교차검증 평균 정확도를 출력하세요.
3. 테스트셋에 대해 confusion matrix와 `classification_report`(정밀도/재현율/F1)를 출력하세요. **지각(1)을 재현율(recall) 기준으로 살펴보면 어떤가요?** — 지각을 "놓치는 것"(재현율이 낮은 것)과 "괜히 경고하는 것"(정밀도가 낮은 것) 중 어느 쪽이 더 나쁠지 생각해보세요.
4. 아래 `my_assignment`에 다가오는 과제 정보를 입력하고, `predict_proba()`로 지각 확률을 예측해보세요.

In [ ]:
# TODO 1: StandardScaler + LogisticRegression 파이프라인을 학습시키세요.


# TODO 2: 5-fold 교차검증 평균 정확도를 출력하세요.


# TODO 3: confusion matrix와 classification_report를 출력하세요.


# TODO 4: 다가오는 과제 정보를 입력하고 지각 확률을 예측해보세요.
my_assignment = {
    "days_before_started": None,   # 예: 2 (마감 2일 전부터 시작 예정)
    "estimated_hours": None,       # 예: 8
    "num_other_deadlines": None,   # 예: 1
    "difficulty": None,            # 예: 4
}